# Tutorial 1 &mdash; One tissue, three technologies

**ASI-FIMSA Workshop 2026 &mdash; spatial omics, hands-on**

Every spatial-omics platform makes the same promise: *molecules, with their coordinates*.
They keep it in very different ways, and the differences decide what questions you can
ask. This Tutorial puts three of them next to each other on human breast carcinoma and
makes you look at the trade-off directly.

| | Platform | What one measurement is | How many genes |
| --- | --- | --- | --- |
| **1** | **Visium** | a 55 µm spot &mdash; a small disc of tissue, several cells | the whole transcriptome, ~36,600 |
| **2** | **Xenium** | one cell | a targeted panel, 313 |
| **3** | **Atera** | one cell | the whole transcriptome, ~18,000 |

Read the table as a diagonal. Visium gives you every gene but not every cell. Xenium
gives you every cell but only the genes someone chose in advance. Atera &mdash; 10x's
pre-release whole-transcriptome chemistry &mdash; is the corner that used not to exist, and
we get to look at it.

**What you will do**

1. Read a real Space Ranger output with a real reader, and meet the `SpatialData`
   object that all three datasets end up in.
2. Read a real Xenium bundle, cluster a square millimetre of it, and draw the cell
   boundaries.
3. Open a prepared Atera Crop and zoom until individual nuclei are visible under the
   segmentation.
4. Put the three side by side at **matched physical scale**, with every number in the
   comparison computed from the data rather than quoted from a brochure.

**What you need**: nothing but a free Colab session. No GPU. Total compute is a few
minutes; most of the elapsed time is downloading ~110 MB.

> A note on honesty before we start. These are **three different specimens** of human
> invasive breast carcinoma, all FFPE, all from 10x Genomics &mdash; not three serial
> sections cut from one block. They are matched by disease and preservation, which is
> close enough that the striking differences you will see are the technology rather than
> the biology, but they are not the same piece of tissue and we will not pretend they are.

## 0. Setup

### 0.1 Install

Colab already ships **numpy**, **pandas**, **matplotlib**, **scipy** and
**scikit-learn**. What is missing is the `spatialdata` family &mdash; the scverse
ecosystem for spatial omics &mdash; plus `scanpy` for the single-cell steps.

Two things about this cell that are worth understanding rather than skipping:

* **The versions are pinned.** `spatialdata`, `spatialdata-io` and `spatialdata-plot`
  are one moving target, not three: the file format and the plotting API change
  together, and mixing versions across them is the single most reliable way to get an
  error message that makes no sense. The Workshop pins all of them to one tested set.
* **`numpy==2.0.2` is pinned too, and it is deliberate.** Colab preloads numpy before
  your first cell runs. If pip *upgrades* numpy mid-session, it swaps the files under a
  module that is already in memory and Colab demands a kernel restart &mdash; halfway
  through the Tutorial. Naming the version Colab already has tells pip to leave it alone.

This takes one to two minutes. You may see a message about restarting the runtime after
the install; you can ignore it &mdash; nothing we install replaces a preloaded module.

In [ ]:
%pip install -q \
    "numpy==2.0.2" \
    "spatialdata==0.8.0" "spatialdata-io==0.7.1" "spatialdata-plot==0.4.1" \
    "scanpy==1.12.3" "igraph==1.0.0" \
    "huggingface_hub==1.28.0" "gdown==6.1.0"

### 0.2 Imports and settings

`spatialdata_plot` looks unused after you import it, and your linter will tell you so.
It is not: importing it **registers a `.pl` accessor** onto every `SpatialData` object,
which is how `sdata.pl.render_images(...)` comes to exist. This is the same trick pandas
uses for `df.plot`.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import spatialdata as sd
import spatialdata_io
import spatialdata_plot  # noqa: F401  -- registers the .pl accessor on SpatialData
import zarr

warnings.filterwarnings("ignore")
sc.settings.verbosity = 0
plt.rcParams["figure.dpi"] = 100

print("spatialdata      :", sd.__version__)
print("spatialdata-io   :", spatialdata_io.__version__)
print("spatialdata-plot :", spatialdata_plot.__version__)
print("scanpy           :", sc.__version__)
print("anndata          :", ad.__version__)
print("numpy            :", np.__version__)
print("zarr             :", zarr.__version__)

---

## 1. Visium &mdash; 55 µm spots, every gene

The Visium slide is a glass slide printed with ~5,000 spots in a honeycomb. Each spot is
**55 µm across**, spaced **100 µm centre to centre**, and carries millions of copies of a
barcode unique to that spot. You stain and image the section in H&E first, then permeabilise
it; mRNA diffuses down onto the spot beneath it, gets reverse-transcribed with the spot's
barcode attached, and the whole library is sequenced. Afterwards the barcode tells you which
spot a transcript came from, and the image tells you where that spot was.

Because the readout is ordinary RNA-seq, you get **the whole transcriptome**: 36,601 genes
here, nobody had to choose them in advance. Because the unit is a 55 µm disc, you do **not**
get cells.

We use the public 10x sample **`V1_Breast_Cancer_Block_A_Section_1`** &mdash; an invasive
ductal carcinoma, 3,798 spots under tissue, processed with Space Ranger 1.1.0 back in 2020.

### 1.1 Download

Two files, ~38 MB:

| File | Size | What it is |
| --- | --- | --- |
| `..._filtered_feature_bc_matrix.h5` | 28 MB | the counts, genes &times; spots |
| `..._spatial.tar.gz` | 10 MB | the H&E images, the spot coordinates and the scale factors |

> **The file we skip here.** The same page offers `..._image.tif`, the full-resolution
> H&E, at **1.8 GB**. The tarball already contains a 2000 &times; 2000 downscaled copy,
> which is plenty for *looking at* the slide and about 100&times; less to fetch, so we
> leave the big one alone in this Tutorial.
>
> Whether you can skip it depends entirely on what you intend to do. Tutorial 03 *does*
> download it: it cuts a small tile of image around every spot and trains a model on them,
> and at 2000 px a whole 55 µm spot is about 15 pixels across — too coarse for a nucleus to
> exist. Judging which files a dataset actually requires, for your question, is a genuinely
> useful skill with these datasets.

In [ ]:
%%bash
set -euo pipefail
BASE=https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1
SAMPLE=V1_Breast_Cancer_Block_A_Section_1

mkdir -p visium && cd visium
for f in ${SAMPLE}_filtered_feature_bc_matrix.h5 ${SAMPLE}_spatial.tar.gz; do
    if [ ! -s "$f" ]; then
        echo "downloading $f ..."
        wget -q --tries=5 --timeout=60 --continue "$BASE/$f"
    else
        echo "$f already present, skipping"
    fi
done
tar -xzf ${SAMPLE}_spatial.tar.gz

echo
echo "--- visium/ ---";         ls -lh
echo
echo "--- visium/spatial/ ---"; ls -lh spatial/

### 1.2 Look at that file listing before you go on

In `visium/spatial/` there is a file called **`tissue_positions_list.csv`**. That is the
spot coordinate table, and its name is a small piece of history.

* Space Ranger **1.x** (2020) wrote **`tissue_positions_list.csv`**, with **no header row**.
  You were expected to know the column order.
* Space Ranger **2.0 and later** writes **`tissue_positions.csv`**, **with** a header.

Same table, two names, two conventions. Data you download today can be either, depending on
when it was processed, and 2020-vintage public datasets like this one are still everywhere in
the literature.

**Why this matters, and why it will not bite us.** `spatialdata_io.visium()` and
`scanpy.read_visium()` both look for either filename and sniff for a header, so if you use a
real reader you never even notice. The trap is only sprung if you read the CSV **by hand**:

```python
pos = pd.read_csv("spatial/tissue_positions_list.csv")   # <-- silently wrong on SR 1.x
```

pandas takes the first *spot* as a header row, every column ends up named after a barcode,
and you have lost one spot and gained a very confusing DataFrame. It does not raise. It is
the sort of bug that eats an afternoon.

**The lesson of this section**: use the reader. It is not laziness &mdash; the reader is
where several years of accumulated knowledge about vendor format drift actually lives.

### 1.3 Read it with the real reader

One line. `spatialdata_io.visium()` finds the counts, the images, the spot table and the
scale factors, checks them against each other, and hands back a `SpatialData` object.

In [ ]:
SAMPLE = "V1_Breast_Cancer_Block_A_Section_1"

visium = spatialdata_io.visium("visium")
visium

### 1.4 The anatomy of a `SpatialData` object

Everything in this Tutorial &mdash; all three platforms &mdash; ends up in an object shaped
like the one printed above. It has exactly five kinds of content, and one idea that ties
them together.

| Slot | Holds | Here |
| --- | --- | --- |
| **Images** | raster pictures (H&E, immunofluorescence) | the hi-res and low-res H&E |
| **Labels** | raster segmentation masks &mdash; one integer per pixel saying which cell it belongs to | *(none for Visium)* |
| **Points** | individual molecules with coordinates | *(none for Visium)* |
| **Shapes** | vector geometry &mdash; circles, polygons | the 3,798 spots, as circles |
| **Tables** | measurements, as `AnnData` &mdash; the familiar cells &times; genes matrix | the count matrix |

And the idea that ties them together: **coordinate systems**. Each element stores its own
coordinates in whatever units it was born with &mdash; the H&E in pixels, the spots in
full-resolution pixels &mdash; plus a **transformation** into one or more shared coordinate
systems. Ask for an element "in coordinate system X" and `spatialdata` applies the
transformation for you. That is the entire reason you can overlay a 2000-pixel image on
3,798 spots recorded against a 20,000-pixel one and have them line up.

Notice the reader made three coordinate systems for us: the native full-resolution one, and
a downscaled one for each image. Let us look at the pieces.

In [ ]:
from spatialdata.transformations import get_transformation

# The reader names elements after the sample, so let us give them short handles.
IMG_HIRES = f"{SAMPLE}_hires_image"
SPOTS = SAMPLE
CS_HIRES = f"{SAMPLE}_downscaled_hires"     # the coordinate system the hi-res image is native in

print("--- Images ---")
for name, img in visium.images.items():
    print(f"  {name:52s} {tuple(img.shape)}  (channels, rows, cols)")

print("\n--- Shapes ---")
spots = visium.shapes[SPOTS]
print(f"  {SPOTS:52s} {len(spots):,} {spots.geometry.iloc[0].geom_type} geometries")
print(f"  columns: {list(spots.columns)}   spot radius: {spots['radius'].iloc[0]:.1f} px")

print("\n--- Tables ---")
vis_table = visium.tables["table"]
print(" ", vis_table)

print("\n--- Coordinate systems ---")
for cs in visium.coordinate_systems:
    print(f"  {cs}")
print("\nHow the spots get into the hi-res image's coordinate system:")
print(" ", get_transformation(spots, to_coordinate_system=CS_HIRES))

The transformation printed at the bottom is a plain scale of about 0.0825 &mdash; the ratio
between the 24,240-pixel full-resolution image and the 2000-pixel one we downloaded.
`spatialdata` carries it so you never have to.

### 1.5 Normalise the counts

Standard scanpy preprocessing, and standard for a reason. A spot that simply captured more
RNA is not "higher" for every gene; normalising every spot to the same total, then `log1p`,
puts spots on a comparable footing. We keep the raw counts in a layer, because raw counts
are what you want for any statistical test later.

In [ ]:
vis_table.layers["counts"] = vis_table.X.copy()      # keep the raw counts

vis_counts_per_spot = np.asarray(vis_table.layers["counts"].sum(axis=1)).ravel()
vis_genes_per_spot = np.asarray((vis_table.layers["counts"] > 0).sum(axis=1)).ravel()
print(f"spots               : {vis_table.n_obs:,}")
print(f"genes               : {vis_table.n_vars:,}")
print(f"median counts / spot: {np.median(vis_counts_per_spot):,.0f}")
print(f"median genes  / spot: {np.median(vis_genes_per_spot):,.0f}")

sc.pp.normalize_total(vis_table, target_sum=1e4)
sc.pp.log1p(vis_table)
print("\nnormalised to 10,000 counts per spot, then log1p")

Look at those two medians. Roughly 21,000 transcripts and 6,000 distinct genes **per spot**
&mdash; far more than you would ever get from a single cell, because a spot is not a single
cell. That depth is exactly what you are buying with the resolution you are giving up.

### 1.6 Put four genes on the tissue

Four markers, chosen so the picture means something to an immunologist:

| Gene | Reads out |
| --- | --- |
| `ERBB2` | HER2 &mdash; the therapeutic target, and amplified in a subset of breast tumours |
| `EPCAM` | epithelium, i.e. where the tumour cells are |
| `COL1A1` | collagen I &mdash; fibroblasts and desmoplastic stroma |
| `PTPRC` | CD45 &mdash; every leukocyte, so: immune infiltration |

`spatialdata-plot` uses a chained, declarative style: each `.pl.render_*` call *adds a
layer*, and `.pl.show()` draws the stack. Passing a **list** of genes to `color=` gives you
one panel per gene, which is exactly what we want.

In [ ]:
MARKERS_VIS = ["ERBB2", "EPCAM", "COL1A1", "PTPRC"]
assert all(g in vis_table.var_names for g in MARKERS_VIS)

(
    visium.pl.render_images(IMG_HIRES)
    .pl.render_shapes(SPOTS, color=MARKERS_VIS, cmap="viridis", fill_alpha=0.9)
    .pl.show(coordinate_systems=CS_HIRES, ncols=2, figsize=(11, 9),
             frameon=False, hspace=0.02)
)
plt.show()

Read the four panels against each other. `EPCAM` and `ERBB2` light up the same territory
&mdash; the tumour epithelium &mdash; while `COL1A1` fills the space *between* those
regions. That anticorrelation is the tumour/stroma architecture of the section, visible
without anyone having annotated anything.

Then look at `PTPRC`. It is low and diffuse almost everywhere, with a few brighter patches.
An immunologist looking at this slide down a microscope would see obvious lymphoid
aggregates. Where are they?

### 1.7 What a 55 µm spot hides

A spot is a disc 55 µm across, so its area is about **2,376 µm²**. A breast epithelial cell
is roughly 10&ndash;20 µm across. The usual quoted figure is **1 to 10 cells per Visium
spot**, depending on tissue density &mdash; and in Section 4 we will stop quoting it and
compute it, using the actual cell density measured on the same disease by the two
single-cell platforms.

The consequence is worth stating plainly, because it is the reason the other two sections of
this Tutorial exist:

* **Every spot is a mixture.** Three T cells among seven tumour cells give you a spot that
  is transcriptionally *mostly tumour*. The T cell signal is real but diluted.
* **You cannot count cells.** "This region has more `PTPRC`" is a statement about
  transcripts per unit area, not about how many leukocytes are there. More CD45 could mean
  more T cells, or the same number of T cells expressing more.
* **You cannot co-localise within a spot.** If a spot has `CD3D` and `CD68`, you cannot tell
  whether a T cell is touching a macrophage or whether they are 50 µm apart.

None of this makes Visium a bad instrument. It makes it an instrument for a particular kind
of question &mdash; and it sets up the obvious next one: what if the unit of measurement
were the cell?

---

## 2. Xenium &mdash; single cells, 313 genes

Xenium never sequences anything. It is **imaging**: padlock probes bind their target mRNA
*in situ*, get amplified into a bright rolling-circle product, and then the instrument
photographs the section over many rounds of fluorescent readout. Each gene has a codeword
&mdash; a pattern of on/off across the rounds &mdash; and decoding that pattern gives you a
transcript with a **sub-micron coordinate**. Cells are segmented from a nuclear stain plus a
boundary stain, and transcripts are assigned to whichever cell they fall in.

The cost of doing it this way is that the number of codewords is finite, so somebody has to
choose the genes in advance. This dataset uses the 313-gene **Breast Cancer Tumor
Microenvironment** panel: 280 pre-designed targets plus 33 custom add-ons.

We use the public **`Xenium_FFPE_Human_Breast_Cancer_Rep1`** bundle: 167,780 cells.

### 2.1 Download six small files

A full Xenium `outs/` folder is several gigabytes, most of it morphology images and the
per-transcript table. We need six files, ~33 MB in total, and we lay them out in the folder
structure the reader expects.

| File | Size | What it is |
| --- | --- | --- |
| `experiment.xenium` | 1.4 KB | the run's metadata: pixel size, panel, software version |
| `cells.parquet` | 3.5 MB | one row per cell: centroid, area, transcript counts |
| `cell_feature_matrix.h5` | 12.1 MB | the counts, features &times; cells |
| `cell_boundaries.parquet` | 8.8 MB | the cell outline polygons |
| `nucleus_boundaries.parquet` | 8.3 MB | the nucleus outline polygons |
| `gene_panel.json` | 154 KB | which genes are on the panel, and why |

In [ ]:
%%bash
set -euo pipefail
BASE=https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep1
PREFIX=Xenium_FFPE_Human_Breast_Cancer_Rep1

mkdir -p xenium && cd xenium
for f in experiment.xenium cells.parquet cell_feature_matrix.h5 \
         cell_boundaries.parquet nucleus_boundaries.parquet gene_panel.json; do
    if [ ! -s "$f" ]; then
        echo "downloading $f ..."
        # note the -O: the reader wants the plain names, not the sample-prefixed ones
        wget -q --tries=5 --timeout=60 -O "$f" "$BASE/${PREFIX}_$f"
    else
        echo "$f already present, skipping"
    fi
done

echo
echo "--- xenium/ ---"; ls -lh

In [ ]:
# What the instrument recorded about this run.
specs = json.loads(Path("xenium/experiment.xenium").read_text())
for key in ["run_name", "preservation_method", "num_cells", "transcripts_per_cell",
            "panel_name", "panel_num_targets_predesigned", "panel_num_targets_custom",
            "pixel_size", "instrument_sn", "analysis_sw_version"]:
    print(f"  {key:32s} {specs[key]}")

### 2.2 One missing file, and an honest workaround

`spatialdata_io.xenium()` opens **`cells.zarr.zip`** unconditionally, and we did not
download it, because it is **315 MB**. So we are going to make an empty one. This deserves
an explanation rather than a magic incantation, because you should know exactly what you are
giving up.

`cells.zarr.zip` is 10x's Xenium Explorer file. In Xenium onboard analysis **1.3 and later**
it carries two things the reader genuinely needs: the encoding that turns internal cell
indices into cell-ID strings, and the **raster segmentation masks** &mdash; a label image
with one integer per pixel.

Look at `analysis_sw_version` printed above: **`Xenium-1.0.1`**. That is *older* than 1.3.0,
and on that branch the reader **never reads an array out of this file**. It asks the zarr
group exactly two membership questions &mdash; "do you contain `polygon_sets`?" and "do you
contain `seg_mask_value`?" &mdash; and for a 1.0.1 bundle the honest answer to both is *no*.
An empty zarr group answers *no* to both. So a 182-byte file tells the reader the truth.

**What we are actually giving up**: the raster segmentation masks, which the reader would
have loaded as `cells_labels` and `nucleus_labels`. We switch those off in the call below
anyway. The segmentation itself is not lost &mdash; `cell_boundaries.parquet` carries the
same segmentation as **vector polygons**, which is both smaller and what we want to draw.

> **This is a Workshop shortcut, not a recipe.** It saves a room full of people 315 MB each
> for a file none of them will read. For your own Xenium run, download the whole `outs/`
> folder and let the reader have everything.

In [ ]:
# Three lines: open a zip-backed zarr store, write one empty group into it, close it.
store = zarr.storage.ZipStore("xenium/cells.zarr.zip", mode="w")
zarr.group(store=store)
store.close()

print("cells.zarr.zip:", Path("xenium/cells.zarr.zip").stat().st_size, "bytes"
      "   (the real one is ~315 MB)")

### 2.3 Read it with the real reader

Same idea as Visium: one call, and everything that follows is `SpatialData`. The keyword
arguments switch off the pieces we did not download &mdash; the raster label masks, the
per-transcript table (~1 GB) and the morphology images. `cells_as_circles=True` additionally
gives us a cheap circle per cell, useful for whole-slide views where drawing 167,780
polygons would be wasteful.

In [ ]:
xenium = spatialdata_io.xenium(
    "xenium",
    cells_labels=False,        # raster cell masks    -- needs the real cells.zarr.zip
    nucleus_labels=False,      # raster nucleus masks -- likewise
    transcripts=False,         # transcripts.parquet  -- ~1 GB, not downloaded
    morphology_mip=False,      # morphology images    -- not downloaded
    morphology_focus=False,
    cells_as_circles=True,     # also give us one circle per cell
)
xenium

It worked, and the shape of the object tells the story: **no Images**, because we did not
download any; **three Shapes** elements, because the segmentation arrived as vector
geometry; and one **Table** of 167,780 cells &times; 313 genes.

### 2.4 313 genes, and 228 things that are not genes

The table has 313 columns. The file on disk has 541. The difference is the part of a Xenium
run that most people never look at, and it is the part that tells you whether to believe the
other 313.

In [ ]:
# Read the matrix again WITHOUT the gene-expression filter, to see everything in it.
all_features = sc.read_10x_h5("xenium/cell_feature_matrix.h5", gex_only=False)
print(all_features.var["feature_types"].value_counts().to_string())
print(f"\ntotal features in the file : {all_features.n_vars}")
print(f"real genes                 : {xenium.tables['table'].n_vars}")
print("\nexamples:")
for kind in pd.unique(all_features.var["feature_types"]):
    names = all_features.var_names[all_features.var["feature_types"] == kind][:3]
    print(f"  {kind:28s} {', '.join(names)}")

Three families of control, each catching a different way the measurement can lie:

* **Negative control probe** (`NegControlProbe_*`) &mdash; a real probe against a sequence
  that is not in the transcriptome. If it lights up, probes are binding where they should
  not. This is the **chemistry** control.
* **Negative control codeword** (`NegControlCodeword_*`) &mdash; a valid codeword that was
  never assigned to any probe. If it lights up, the decoder is inventing calls out of noise.
  This is the **decoding** control.
* **Blank codeword** (`BLANK_*`) &mdash; codewords left unused in the codebook, for the same
  reason.

Together they give you a per-cell false-positive rate measured on your own slide. There is
no equivalent in a sequencing-based assay, and it is one of the quiet advantages of imaging.

In [ ]:
ctrl = all_features[:, all_features.var["feature_types"] != "Gene Expression"].X.sum()
real = all_features[:, all_features.var["feature_types"] == "Gene Expression"].X.sum()
print(f"transcripts called as real genes : {real:,.0f}")
print(f"transcripts called as controls   : {ctrl:,.0f}")
print(f"\ncontrol fraction: {100 * ctrl / (ctrl + real):.2f}%   (well under 1% is healthy)")

del all_features   # 541 x 167,780 is not small; let it go

### 2.5 Coordinate systems, and a unit surprise

Before we crop, one thing that will bite you if you do not check it. Print the transformation
that maps the cell polygons into the `"global"` coordinate system.

In [ ]:
from spatialdata.transformations import Identity, get_transformation, set_transformation

cell_polys = xenium.shapes["cell_boundaries"]
print("cell_boundaries -> global :", get_transformation(cell_polys, to_coordinate_system="global"))
print("intrinsic bounds (x0, y0, x1, y1):", np.round(cell_polys.total_bounds, 1))
print("pixel size from experiment.xenium:", specs["pixel_size"], "µm per pixel")
print("\n1 / pixel_size =", round(1 / specs["pixel_size"], 6))

The scale factor is `1 / 0.2125` &mdash; so the polygons are stored in **micrometres**, and
`"global"` is in **camera pixels**. That is the sensible choice for `spatialdata_io`, because
`"global"` is where the morphology images live and images are naturally indexed in pixels.
It is *not* what you want for a workshop about physical scale, and it is not what the Atera
object in Section 3 uses.

Rather than divide by 0.2125 everywhere and hope, we add a **second coordinate system in
micrometres**. Because the stored coordinates are already micrometres, the transformation is
the identity, and this is three lines. Everything downstream &mdash; the crop, the plots, the
axis labels &mdash; is then in µm.

This is what coordinate systems are *for*: not bookkeeping, but a place to say "and here is
the same data in the frame I want to think in".

In [ ]:
UM = "micrometres"
for name in xenium.shapes:
    set_transformation(xenium.shapes[name], Identity(), to_coordinate_system=UM)

print(xenium.coordinate_systems)

### 2.6 Crop to one square millimetre

167,780 cells is more than we want to cluster in a live session, and a whole slide at cell
resolution is unreadable on a laptop screen anyway. `sdata.query.bounding_box()` cuts a
window out of **every** element at once &mdash; images, shapes and table together &mdash;
which is the whole point of keeping them in one object.

We take a 1 mm &times; 1 mm square containing a duct and the stroma around it.

In [ ]:
X0, Y0, SIDE = 5500.0, 3000.0, 1000.0      # micrometres

xen_crop = xenium.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[X0, Y0],
    max_coordinate=[X0 + SIDE, Y0 + SIDE],
    target_coordinate_system=UM,
    filter_table=True,                      # keep only the rows for cells inside the window
)
xen_crop

Note that `cell_boundaries` comes back with slightly more geometries than the table has rows.
Those are cells whose polygon overlaps the window but whose centroid sits outside it: the
geometric query keeps the polygon, `filter_table=True` drops the row. They will render as
"NA" in the plot below. It is not a bug, it is what an edge is.

### 2.7 Cluster it

The standard single-cell recipe, and it is standard here for the same reasons it is standard
in scRNA-seq: normalise for how much RNA each cell gave up, log, scale so no gene dominates,
compress to principal components, build a k-nearest-neighbour graph on those, and cut the
graph into communities with **Leiden**.

Two Xenium-specific notes:

* We normalise to a target of 100 counts, not 10,000. With 313 genes and a median of ~160
  transcripts per cell, 10,000 would be a wild extrapolation.
* We pass `flavor="igraph"`, which uses the Leiden implementation inside `python-igraph`.
  The alternative, `leidenalg`, is an extra dependency we do not need.

About 20 seconds on ~6,000 cells.

In [ ]:
xen_table = xen_crop.tables["table"]
xen_table.layers["counts"] = xen_table.X.copy()
print(f"cells in the crop: {xen_table.n_obs:,}")

sc.pp.normalize_total(xen_table, target_sum=100)
sc.pp.log1p(xen_table)
xen_table.layers["lognorm"] = xen_table.X.copy()
sc.pp.scale(xen_table, max_value=10)

sc.tl.pca(xen_table, n_comps=30)
sc.pp.neighbors(xen_table, n_neighbors=15, n_pcs=30)
sc.tl.leiden(xen_table, resolution=0.6, flavor="igraph", n_iterations=2, key_added="leiden")

xen_table.X = xen_table.layers["lognorm"]        # put the log-normalised values back for plotting

# Leiden labels are strings, so they sort as 0, 1, 10, 11, 2 ... -- put them back in order.
xen_table.obs["leiden"] = xen_table.obs["leiden"].cat.reorder_categories(
    sorted(xen_table.obs["leiden"].cat.categories, key=int)
)
print(f"\n{xen_table.obs['leiden'].nunique()} clusters")
print(xen_table.obs["leiden"].value_counts().sort_index().to_string())

Clusters are numbers, and numbers are not biology. The honest next step is to ask what each
cluster expresses, using markers you already trust.

In [ ]:
DOT_MARKERS = {
    "Tumour epithelial": ["EPCAM", "KRT8", "ERBB2", "ESR1"],
    "Proliferating": ["MKI67", "TOP2A"],
    "Myoepithelial": ["KRT14", "KRT5", "ACTA2"],
    "Fibroblast": ["POSTN", "LUM", "PDGFRB"],
    "Endothelial": ["PECAM1", "VWF"],
    "T cell": ["PTPRC", "CD3D", "CD8A", "IL7R"],
    "B / plasma": ["MS4A1", "CD79A", "MZB1"],
    "Myeloid": ["CD68", "LYZ", "ITGAX"],
    "Mast": ["TPSAB1"],
}
sc.pl.dotplot(xen_table, DOT_MARKERS, groupby="leiden", standard_scale="var",
              figsize=(11, 4.5), show=False)
plt.show()

### 2.8 Draw the cell boundaries

One wrinkle first, and it is a nice illustration of how `SpatialData` links things. We asked
the reader for `cells_as_circles=True`, so the table declares that it annotates the
**`cell_circles`** element. If we ask to colour `cell_boundaries` by `leiden`, plotting
correctly refuses: as far as the object is concerned, nothing connects that table to those
polygons.

`set_table_annotates_spatialelement()` re-points the link. The cell IDs are the same in both,
so the join is valid &mdash; we are correcting the bookkeeping, not inventing a relationship.

In [ ]:
xen_table.obs["region"] = pd.Categorical(["cell_boundaries"] * xen_table.n_obs)
xen_crop.set_table_annotates_spatialelement(
    "table", region="cell_boundaries", region_key="region", instance_key="cell_id"
)

(
    xen_crop.pl.render_shapes("cell_boundaries", color="leiden")
    .pl.show(
        coordinate_systems=UM,
        figsize=(8, 7),
        title=f"Xenium — 1 mm² crop, {xen_table.n_obs:,} cells, Leiden clusters",
    )
)
plt.show()

Stop and look at this properly, because it is the payoff of the whole section.

Every one of those outlines is **a cell**, with its own 313-gene expression profile. You can
see the duct wall as a ring of cells one or two thick. You can see the stroma between ducts
as a different, elongated cell shape with a different cluster identity. You can see immune
cells as small polygons scattered through the stroma and, in places, pressed against the duct
edge.

Ask a question here that Visium could not answer: **are the T cells inside the duct or around
it?** On this picture you answer it by looking. On the Visium panel in Section 1, a single
55 µm spot would have covered the duct wall *and* its neighbouring stroma, and averaged them.

And now the price. Every cell here has an expression profile of exactly **313 genes**. If the
gene you care about is not on the panel, it does not exist in this dataset, and no reanalysis
will conjure it. Which brings us to the third technology.

---

## 3. Atera &mdash; single cells, ~18,000 genes

**Atera** is 10x's pre-release whole-transcriptome in-situ chemistry: the single-cell
resolution of Xenium, but with the panel opened up from a few hundred targets to
**18,028**. It is the corner of the trade-off table that used to be empty.

**Be clear-eyed about what this dataset is.** It is a *preview*: the run below reports
`chemistry_version: "Atera v1"` on a **Gen2 prototype instrument**, and is a public
demonstration bundle rather than a released product. Sensitivity, specificity and
segmentation are all subject to change. What it is genuinely good for &mdash; and why it is
in this Workshop &mdash; is showing you what the next point on the curve looks like.

The vendor bundle is ~65 GB, which is not something a room of Colab sessions is going to
download. Two **Staged datasets** were prepared for the Workshop instead:

| Staged dataset | Size | What is in it |
| --- | --- | --- |
| `atera_wholeslide_cells.h5ad` | 19.9 MB | all **170,057** cells: aligned centroids, the vendor's clustering, a UMAP, annotated cell types, and a 69-gene marker panel as the expression matrix. No images. |
| `atera_crop.zarr.zip` | 18.0 MB | the **Crop**: a 2 mm square with H&E at two resolutions, per-cell boundary polygons and the annotated table, all in one `SpatialData` object. |

The cell types were assigned from the vendor's clustering by a human reviewing marker
evidence cluster by cluster, not by an automatic label transfer. The vocabulary you will see
&mdash; *Tumour epithelial, Proliferating tumour, Myoepithelial, Fibroblast, Endothelial,
Perivascular, T cell, Dendritic cell, Macrophage, Plasma cell, Mast, Mixed (plasma+mast),
Unassigned* &mdash; is that reviewer's, and `Unassigned` means exactly what it says.

### 3.1 Fetch the Staged datasets

They live in a public Hugging Face dataset repository. The helper below tries, in order:
a local override (used by presenters and for offline testing), Hugging Face, then Google
Drive as a mirror &mdash; and if all three fail it says so in words rather than in a
traceback.

In [ ]:
HF_REPO = "xiao233333/asi-fimsa-workshop-2026"
GDRIVE_FOLDER = "1ELxQjcswMcO7w4N6Z74u_2Yt3F5UWHbm"
STAGED_DIR = Path("atera")
STAGED_DIR.mkdir(exist_ok=True)


def fetch_staged(filename: str) -> Path:
    '''Fetch one Staged dataset: local override, then Hugging Face, then Google Drive.'''
    out = STAGED_DIR / filename
    if out.exists() and out.stat().st_size > 0:
        print(f"{filename}: already here ({out.stat().st_size / 1e6:.1f} MB)")
        return out

    # 1. Local override -- set WORKSHOP_DATA_DIR to a folder holding the files.
    override = os.environ.get("WORKSHOP_DATA_DIR")
    if override:
        src = Path(override) / filename
        if not src.exists():
            raise FileNotFoundError(f"WORKSHOP_DATA_DIR is set to {override} but {filename} is not in it")
        import shutil
        shutil.copyfile(src, out)
        print(f"{filename}: copied from WORKSHOP_DATA_DIR ({out.stat().st_size / 1e6:.1f} MB)")
        return out

    # 2. Hugging Face (primary).
    try:
        from huggingface_hub import hf_hub_download
        got = hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset",
                              local_dir=str(STAGED_DIR))
        print(f"{filename}: from Hugging Face ({Path(got).stat().st_size / 1e6:.1f} MB)")
        return Path(got)
    except Exception as hf_error:
        print(f"Hugging Face did not work ({type(hf_error).__name__}); trying the Google Drive mirror.")

    # 3. Google Drive (mirror).
    try:
        import gdown
        gdown.download_folder(id=GDRIVE_FOLDER, output=str(STAGED_DIR), quiet=True, use_cookies=False)
        if out.exists():
            print(f"{filename}: from the Google Drive mirror ({out.stat().st_size / 1e6:.1f} MB)")
            return out
    except Exception as drive_error:
        print(f"The Google Drive mirror did not work either ({type(drive_error).__name__}).")

    print(
        "\n"
        "-------------------------------------------------------------------\n"
        f"Could not fetch '{filename}'.\n"
        "\n"
        "This is a download problem, not a problem with your session, and\n"
        "nothing you have run so far is affected. Please tell a presenter --\n"
        "they have the files on a USB stick.\n"
        "\n"
        "If you would rather try yourself:\n"
        f"  Hugging Face : https://huggingface.co/datasets/{HF_REPO}\n"
        f"  Google Drive : https://drive.google.com/drive/folders/{GDRIVE_FOLDER}\n"
        "Download the file, drag it into the Colab file browser on the left\n"
        "into a folder called 'atera', and re-run this cell.\n"
        "-------------------------------------------------------------------"
    )
    raise RuntimeError(f"could not fetch {filename} -- see the message above")


crop_zip = fetch_staged("atera_crop.zarr.zip")
cells_h5ad = fetch_staged("atera_wholeslide_cells.h5ad")

### 3.2 Open them

`atera_wholeslide_cells.h5ad` is a plain `AnnData`, so `anndata.read_h5ad()` opens it.

The Crop needs one extra step. It is a `SpatialData` object stored as a **zipped** zarr
directory, and as of `spatialdata` 0.8 `read_zarr()` **cannot open a `.zarr.zip` in place**
&mdash; it treats the path as a directory, finds no group, and raises. So: unzip first, then
read the directory. The object even ships instructions to that effect in its own metadata,
which we print below &mdash; that note was written against `spatialdata` 0.7.3 and asks for a
re-test on 0.8; consider it re-tested, because the behaviour has not changed.

In [ ]:
%%bash
set -euo pipefail
cd atera
if [ ! -d atera_crop.zarr ]; then
    unzip -q atera_crop.zarr.zip -d atera_crop.zarr
fi
echo "atera_crop.zarr/ contains:"
ls atera_crop.zarr

In [ ]:
whole = ad.read_h5ad(cells_h5ad)
crop = sd.read_zarr("atera/atera_crop.zarr")

print(whole)
print()
print(crop)
print()
print("what the Crop says about itself:")
print("  ", crop.attrs["atera"]["read_instructions"])

### 3.3 The whole slide, one dot per cell

170,057 cells, coloured by the annotated cell type. Two plotting details that matter at this
size: `s=0.3` so a dot is roughly one cell rather than a blob, and `rasterized=True` so the
figure is stored as pixels rather than as 170,057 vector objects &mdash; the difference
between a notebook that scrolls and one that does not.

In [ ]:
cell_types = list(whole.obs["cell_type"].cat.categories)
palette = dict(zip(cell_types, whole.uns["cell_type_colors"]))
xy_whole = whole.obsm["spatial"]

fig, ax = plt.subplots(figsize=(8.5, 11))
for ct in cell_types:
    keep = (whole.obs["cell_type"] == ct).to_numpy()
    ax.scatter(xy_whole[keep, 0], xy_whole[keep, 1], s=0.3, c=palette[ct],
               linewidths=0, rasterized=True, label=f"{ct}  ({keep.sum():,})")
ax.set_aspect("equal")
ax.invert_yaxis()                       # image convention: y increases downwards
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm)")
ax.set_title(f"Atera whole slide — {whole.n_obs:,} cells, {len(cell_types)} annotated types")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
          markerscale=18, fontsize=9)
plt.show()

This is a whole tumour section drawn one cell at a time.

The red masses are ducts filled with tumour epithelium, and if you look at their edges you
can pick out the thin orange line of **myoepithelial** cells that still surrounds many of
them &mdash; the histological signature of *ductal carcinoma in situ*, and a genuinely
important distinction from invasion. The blue **T cells** are not uniform: look at the lower
third of the section, where they crowd around and between the ducts, versus the upper
regions where the same ducts sit in comparatively quiet stroma. That is spatially
heterogeneous immune infiltration, on one slide, without staining anything extra.

### 3.4 The same cells without their coordinates

The expression side of the same object. `X_umap` was computed by the vendor pipeline on the
full ~18,000-gene matrix, so this is an honest whole-transcriptome embedding, not one derived
from the 69 markers we ship.

In [ ]:
umap_xy = whole.obsm["X_umap"]

fig, ax = plt.subplots(figsize=(7.5, 6))
for ct in cell_types:
    keep = (whole.obs["cell_type"] == ct).to_numpy()
    ax.scatter(umap_xy[keep, 0], umap_xy[keep, 1], s=0.3, c=palette[ct],
               linewidths=0, rasterized=True, label=ct)
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_title("Atera — expression space")
ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
          markerscale=18, fontsize=8)
plt.show()

Worth a moment on what just happened. The previous figure and this one contain **the same
170,057 points**. One is arranged by where the cells were; the other by what they were
expressing. Spatial omics is the discipline of holding both at once &mdash; and of noticing
when a cluster that looks tidy in UMAP turns out to be two different places on the slide.

### 3.5 The same markers, at cell resolution

Section 1 put `ERBB2`, `COL1A1` and `PTPRC` on 3,798 Visium spots. Here they are again on
170,057 cells, with `EPCAM` added for contrast.

We draw them as **detection maps**: every cell in pale grey, and every cell carrying **at
least one transcript** of that gene in colour. That is not laziness &mdash; two cells from
now you will see why it is the honest choice for this dataset.

In [ ]:
GENE_COLOURS = [("EPCAM", "#b2182b"), ("ERBB2", "#762a83"),
                ("PTPRC", "#1f6fb4"), ("COL1A1", "#8c6d3f")]

fig, axes = plt.subplots(1, 4, figsize=(17, 7))
for ax, (gene, colour) in zip(axes, GENE_COLOURS):
    counts = np.asarray(whole[:, gene].X).ravel()
    detected = counts > 0
    ax.scatter(xy_whole[:, 0], xy_whole[:, 1], s=0.3, c="0.87",
               linewidths=0, rasterized=True)                  # every cell, for context
    ax.scatter(xy_whole[detected, 0], xy_whole[detected, 1], s=1.4, c=colour,
               linewidths=0, rasterized=True)                  # cells that detected the gene
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title(f"{gene}\n{detected.sum():,} cells ({100 * detected.mean():.0f}%)", fontsize=11)
fig.suptitle("Cells carrying at least one transcript of each gene", y=1.0)
plt.tight_layout()
plt.show()

Compare these with the Visium panels in Section 1. `EPCAM` fills the ducts, `COL1A1` fills
the space between them, and `PTPRC` concentrates in the lower third of the section and in two
or three dense knots that look very much like lymphoid aggregates. The Visium `PTPRC` panel
was a soft haze over the whole slide; this one resolves into individual CD45-positive cells
you could count, cluster, or measure the distance from a duct.

### 3.5.1 Now the catch, and it is a real one

Look at the percentages in those titles. `EPCAM` was detected in about **47%** of cells but
`ERBB2` in about **4%** &mdash; and `ERBB2` is a tumour gene on a slide that is 45% tumour.
Are the tumour cells not expressing it?

They are. We are mostly not *seeing* it. A whole-transcriptome in-situ assay spreads a fixed
budget of detected transcripts across a far larger number of genes.

In [ ]:
atera_depth = float(np.median(whole.obs["transcript_counts"]))
atera_genes = int(whole.uns["atera"]["n_targets_full_panel"])
xen_depth = float(np.median(xenium.tables["table"].obs["transcript_counts"]))
xen_genes = int(xenium.tables["table"].n_vars)

print(f"Xenium : {xen_depth:>7,.0f} transcripts per cell over {xen_genes:>6,} genes"
      f"   ->  {xen_depth / xen_genes:.3f} counts per gene per cell")
print(f"Atera  : {atera_depth:>7,.0f} transcripts per cell over {atera_genes:>6,} genes"
      f"   ->  {atera_depth / atera_genes:.3f} counts per gene per cell")
print(f"\nper gene, Atera is about "
      f"{(xen_depth / xen_genes) / (atera_depth / atera_genes):.0f}x sparser on this tissue")

Many times more transcripts per cell, spread over nearly sixty times more genes. Per gene the
whole-transcriptome assay comes out several times sparser &mdash; and the real gap is wider
still, because the 313 genes on a Xenium panel were *chosen* to be abundant and informative,
while a whole-transcriptome assay must also carry the seventeen thousand genes nobody would
have picked.

So the honest summary of this technology is not "Xenium, but with every gene". It is:

* **You no longer have to choose the genes.** Nothing is invisible by construction, and you
  can ask a question you had not thought of when you put the slide on the instrument. That is
  the single biggest limitation of a targeted panel, and it is gone.
* **Any one gene is sparse.** A single-cell, single-gene value is mostly zero. You do the
  biology on aggregates &mdash; cluster the cells, then compare clusters &mdash; rather than
  on individual measurements.
* **This is preview chemistry.** Sensitivity is precisely the number you would expect to move
  most between a preview and a released product, so do not read those percentages as the
  platform's final word.

It is also why the cell types on this slide were assigned by **clustering across the whole
transcriptome**, not by thresholding single markers. Sparsity is not a defect to hide; it is
a property to design the analysis around.

### 3.6 The Crop &mdash; cells on their own H&E

The whole-slide view has no image behind it. The Crop does: a 2 mm square with the H&E, the
per-cell boundary polygons, and the table, in one object and one coordinate system
(`"global"`, in **micrometres** &mdash; unlike Xenium's pixel-based `"global"`, which is why
we made a micrometre system for that one).

One word about how this gets drawn, because it matters in a moment. Above a few thousand
shapes `spatialdata-plot` quietly switches from matplotlib to a **`datashader`** backend,
which rasterises the geometry onto the figure canvas instead of drawing every polygon as a
vector. At 16,006 cells across 2 mm that is faster and you cannot tell the difference. In the
zoom that follows, you very much can.

In [ ]:
crop_table = crop.tables["table"]
print(f"cells in the Crop : {crop_table.n_obs:,}")
print(f"genes shipped     : {crop_table.n_vars}")
print("\nelements:")
for name, img in crop.images.items():
    print(f"  image  {name:12s} {tuple(img.shape)}")
for name, shp in crop.shapes.items():
    print(f"  shapes {name:12s} {len(shp):,} polygons")
print("\ncoordinate systems:", crop.coordinate_systems)

In [ ]:
(
    crop.pl.render_images("he")
    .pl.render_shapes("cell_boundaries", color="cell_type", fill_alpha=0.75,
                      outline_alpha=0)
    .pl.show(coordinate_systems="global", figsize=(10, 8),
             title=f"Atera Crop — 2 mm², H&E with {crop_table.n_obs:,} annotated cells")
)
plt.show()

This single figure is the argument for the whole field.

The pink and purple underneath is a slide any pathologist has been reading for a century.
The colours on top are the molecular identity of every single cell in it. The orange rims
around the red duct interiors are myoepithelial cells &mdash; identified by `KRT14`, `KRT5`
and `ACTA2`, not by their shape &mdash; and their presence is what makes those ducts *in
situ* rather than invasive. The blue T cells are in the stroma, at the duct margin, and in a
few places apparently within the epithelium.

### 3.7 Zoom until you can see nuclei

The Crop ships a second image, `he_zoom`: a 300 µm window at the H&E's **native**
0.2738 µm per pixel. Same coordinate system, so we simply change the axis limits.

Here we **do** ask for `method="matplotlib"` on the boundaries, and this is the reason. A
rasterising backend draws at the resolution of the whole rendered extent; zoom the axes into
a twentieth of it afterwards and you are magnifying a low-resolution image, so the cells come
back as blocks and the thin black outlines smear into mush. Drawing real vector polygons
costs a second or two and stays crisp at any zoom. **Rule of thumb: datashader for the
overview, matplotlib whenever you are going to look closely or want per-cell outlines.**

In [ ]:
zoom = crop.attrs["atera"]["crop_window"]["he_zoom"]
zx, zy, zs = zoom["x0_um"], zoom["y0_um"], zoom["size_um"]

# Count the cells in the window from the table rather than trusting a metadata field.
centroids = crop_table.obsm["spatial"]
in_zoom = ((centroids[:, 0] >= zx) & (centroids[:, 0] < zx + zs)
           & (centroids[:, 1] >= zy) & (centroids[:, 1] < zy + zs))
print(f"he_zoom window: {zs:.0f} µm square at ({zx:.0f}, {zy:.0f}) µm, "
      f"{in_zoom.sum()} cells")

fig, axes = plt.subplots(1, 2, figsize=(13, 6.6))
(
    crop.pl.render_images("he_zoom")
    .pl.show(coordinate_systems="global", ax=axes[0],
             title=f"H&E, {zs:.0f} µm across, native resolution")
)
(
    crop.pl.render_images("he_zoom")
    .pl.render_shapes("cell_boundaries", color="cell_type", fill_alpha=0.35,
                      outline=True, outline_alpha=0.9, outline_color="black",
                      outline_width=0.5, method="matplotlib")
    .pl.show(coordinate_systems="global", ax=axes[1],
             title="the same field, with Atera cells", legend_loc=None)
)
for ax in axes:
    ax.set_xlim(zx, zx + zs)
    ax.set_ylim(zy + zs, zy)            # y downwards, as in the image
plt.tight_layout()
plt.show()

Left panel: nuclei. Individual ones, with the pink collagen band across the top and a
scatter of small dark cells below it. Right panel: the same nuclei, each inside a polygon,
each polygon carrying an expression profile that spans about eighteen thousand genes
(colours as in the previous figure).

Look at the band of blue polygons through the middle. Those are T cells, in stroma, adjacent
to a duct whose cells are red. This field is 300 µm on a side. In Section 1 the whole of it
&mdash; T cells, stroma, duct edge and all &mdash; would have been covered by **about a dozen
Visium spots** (they sit on a 100 µm lattice), and the T cells would have shown up as a
slightly raised `PTPRC` value in two or three of them.

That is the resolution argument, and the next section makes it quantitative.

---

## 4. The three side by side

Everything below is **computed from the three objects still in memory**. Nothing is quoted.

In [ ]:
def bbox_mm2(xy: np.ndarray) -> float:
    '''Area of the bounding box of a set of coordinates, in mm-squared.'''
    return float(np.ptp(xy[:, 0]) * np.ptp(xy[:, 1]) / 1e6)     # np.ptp, never .ptp() -- numpy 2.0


def equivalent_diameter_um(areas) -> float:
    '''Diameter of a circle with the median area, in micrometres.'''
    return float(np.sqrt(4 * np.median(np.asarray(areas)) / np.pi))


# Visium spot coordinates are in full-resolution pixels; the scale factors give us µm.
scalefactors = json.loads(Path("visium/spatial/scalefactors_json.json").read_text())
VIS_UM_PER_PX = 55.0 / scalefactors["spot_diameter_fullres"]     # a spot IS 55 µm, by design
vis_xy_um = vis_table.obsm["spatial"] * VIS_UM_PER_PX

xen_table_full = xenium.tables["table"]
xen_xy_um = xen_table_full.obsm["spatial"]                       # already µm

comparison = pd.DataFrame(
    [
        {
            "Platform": "Visium",
            "Unit measured": "55 µm spot",
            "Unit size (µm across)": 55.0,
            "Genes the platform measured": vis_table.n_vars,
            "Genes in the object we loaded": vis_table.n_vars,
            "Units on the slide": vis_table.n_obs,
            "Area surveyed (mm²)": bbox_mm2(vis_xy_um),
            "Units per mm²": vis_table.n_obs / bbox_mm2(vis_xy_um),
            "Median counts per unit": float(np.median(vis_counts_per_spot)),
        },
        {
            "Platform": "Xenium",
            "Unit measured": "cell",
            "Unit size (µm across)": equivalent_diameter_um(xen_table_full.obs["cell_area"]),
            "Genes the platform measured": xen_table_full.n_vars,
            "Genes in the object we loaded": xen_table_full.n_vars,
            "Units on the slide": xen_table_full.n_obs,
            "Area surveyed (mm²)": bbox_mm2(xen_xy_um),
            "Units per mm²": xen_table_full.n_obs / bbox_mm2(xen_xy_um),
            "Median counts per unit": float(np.median(xen_table_full.obs["transcript_counts"])),
        },
        {
            "Platform": "Atera",
            "Unit measured": "cell",
            "Unit size (µm across)": equivalent_diameter_um(whole.obs["cell_area"]),
            "Genes the platform measured": int(whole.uns["atera"]["n_targets_full_panel"]),
            "Genes in the object we loaded": whole.n_vars,
            "Units on the slide": whole.n_obs,
            "Area surveyed (mm²)": bbox_mm2(xy_whole),
            "Units per mm²": whole.n_obs / bbox_mm2(xy_whole),
            "Median counts per unit": float(np.median(whole.obs["transcript_counts"])),
        },
    ]
).set_index("Platform")

comparison.round(1)

Four rows of that table are worth saying out loud.

* **Unit size.** 55 µm against roughly 13 µm and 9 µm. In *area* that is a factor of about
  17 and 37 &mdash; resolution differences look small in a column of numbers and enormous on
  a slide.
* **Genes.** 36,601, then 313, then 18,028. The 313 is not a smaller version of the 36,601;
  it is a different, hand-chosen 313.
* **Median counts per unit.** ~21,000 for a Visium spot, ~160 for a Xenium cell. A spot is
  deep because it is many cells and sequencing is deep; a cell is shallow because it is one
  cell and each transcript had to be individually imaged. Per gene per cell, the two are much
  closer than the raw numbers suggest.
* **Genes we actually loaded.** For Atera this is 69, not 18,028. The Staged dataset ships a
  marker panel to keep the Workshop download under 40 MB. The platform measured all of them;
  we are carrying a slice.

### 4.1 How many cells are under a Visium spot?

Section 1 promised to compute this rather than quote it. We now have two independent measures
of cell density on the same disease, so we can just multiply.

In [ ]:
SPOT_AREA_UM2 = np.pi * (55 / 2) ** 2
print(f"a Visium spot covers {SPOT_AREA_UM2:,.0f} µm² of tissue\n")

for platform in ["Xenium", "Atera"]:
    density_per_mm2 = comparison.loc[platform, "Units per mm²"]
    cells_per_spot = density_per_mm2 * SPOT_AREA_UM2 / 1e6
    print(f"  at the cell density {platform} measures "
          f"({density_per_mm2:,.0f} cells/mm²): {cells_per_spot:.1f} cells per spot")

print("\n(the two disagree because they segment cells differently -- see the next figure --")
print(" and because a bounding box includes empty slide. The honest answer is 'several'.)")

### 4.2 The same square of tissue, three times

The figure below shows a **500 µm &times; 500 µm** square from each dataset, drawn at the
**same number of micrometres per inch**. No panel is zoomed relative to another. This is what
the three technologies would look like if you laid them on the same bench.

In [ ]:
from matplotlib.patches import Circle

SIDE_UM = 500.0


def window_at(cx: float, cy: float) -> tuple[float, float]:
    return cx - SIDE_UM / 2, cy - SIDE_UM / 2


win_vis = window_at(float(np.median(vis_xy_um[:, 0])), float(np.median(vis_xy_um[:, 1])))
win_xen = window_at(6000.0, 3500.0)          # inside the Xenium crop, on a duct
win_ate = window_at(4250.0, 9400.0)          # inside the Atera Crop, on a duct

FILL, EDGE = "#a8c4de", "#25506e"
fig, axes = plt.subplots(1, 3, figsize=(15, 5.8))

# --- Visium: draw each spot at its true 55 µm diameter -----------------------
ax = axes[0]
inside = ((vis_xy_um[:, 0] >= win_vis[0]) & (vis_xy_um[:, 0] < win_vis[0] + SIDE_UM)
          & (vis_xy_um[:, 1] >= win_vis[1]) & (vis_xy_um[:, 1] < win_vis[1] + SIDE_UM))
for x, y in vis_xy_um[
    (np.abs(vis_xy_um[:, 0] - (win_vis[0] + SIDE_UM / 2)) < SIDE_UM / 2 + 60)
    & (np.abs(vis_xy_um[:, 1] - (win_vis[1] + SIDE_UM / 2)) < SIDE_UM / 2 + 60)
]:
    ax.add_patch(Circle((x, y), 27.5, facecolor=FILL, edgecolor=EDGE, lw=0.9))
ax.set_title(f"Visium\n{int(inside.sum())} spots", fontsize=11)
ax.set_xlim(win_vis[0], win_vis[0] + SIDE_UM)
ax.set_ylim(win_vis[1] + SIDE_UM, win_vis[1])

# --- Xenium and Atera: the real segmentation polygons ------------------------
for ax, polys, win, name in [
    (axes[1], xenium.shapes["cell_boundaries"], win_xen, "Xenium"),
    (axes[2], crop.shapes["cell_boundaries"], win_ate, "Atera"),
]:
    sub = polys.cx[win[0]:win[0] + SIDE_UM, win[1]:win[1] + SIDE_UM]
    sub.plot(ax=ax, facecolor=FILL, edgecolor=EDGE, lw=0.4)
    ax.set_title(f"{name}\n{len(sub):,} cells", fontsize=11)
    ax.set_xlim(win[0], win[0] + SIDE_UM)
    ax.set_ylim(win[1] + SIDE_UM, win[1])

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    x_left, y_bottom = ax.get_xlim()[0], ax.get_ylim()[0]
    ax.plot([x_left + 40, x_left + 140], [y_bottom - 45] * 2, lw=3.5, c="k", clip_on=False)
    ax.text(x_left + 90, y_bottom - 60, "100 µm", ha="center", va="bottom", fontsize=9)

fig.suptitle("The same 500 × 500 µm of breast tumour, at matched physical scale", y=1.02)
plt.tight_layout()
plt.show()

Three dozen discs against a couple of thousand cells, at the same scale. That is the whole
argument, and there is no way to present it that makes the Visium panel look finer.

One difference between the two right-hand panels is worth naming, because it is a
segmentation artefact and not biology. Xenium's default cell boundaries are made by
**expanding outwards from each nucleus** until neighbours meet, so they tile the plane with
no gaps. The Atera Crop was segmented from a **boundary stain**, so its polygons follow real
cell membranes and leave genuine extracellular space between them. Neither is wrong; they
answer "where is this cell?" differently, and if you ever compare cell areas across
platforms, this is the first thing to check.

### 4.3 So which one should you use?

There is no winner, only a question you are trying to answer.

| | **Visium** | **Xenium** | **Atera** |
| --- | --- | --- | --- |
| **Buys you** | every gene, no panel design, deep counts, cheap and mature | true single cells, sub-cellular transcript positions, on-slide error controls | true single cells **and** every gene |
| **Costs you** | resolution &mdash; every measurement is a mixture of cells | the panel: an unmeasured gene is unmeasurable, forever | pre-release chemistry, and everything that implies |
| **Good for** | discovery when you do not yet know what to look for; regional questions | testing a hypothesis you can already name in genes; cell-cell contact | both at once &mdash; when you can get on the instrument |

Some practical guidance you can take away:

* **Do not use Visium to count immune cells.** Use it to find the regions where the immune
  signal is, then follow up at cell resolution.
* **Design a Xenium panel with the analysis in mind.** You need the markers that separate the
  cell types you want *and* the genes for the biology you want to measure. That is more genes
  than people expect.
* **Deconvolution is not resolution.** You can estimate cell-type proportions per Visium spot
  from a reference; that is genuinely useful and it is still an estimate of a mixture, not a
  measurement of a cell.
* **Segmentation is a modelling choice, not ground truth**, and it is where a surprising
  amount of downstream disagreement comes from.

### Where this Tutorial sits

You now have three `SpatialData` objects and know how they are put together. **Tutorial 2**
takes the Atera Crop and asks which cells sit next to which &mdash; neighbourhoods, niches,
and whether the immune infiltrate is organised or scattered. **Tutorial 3** goes back to the
Visium sample and asks how much of this you could have predicted from the H&E image alone.

---

## 5. Exercises

Each of these is a small edit to a cell above, followed by re-running from that cell down.

**1. Change the marker.** In Section 1.6, swap `PTPRC` for `MS4A1` (CD20, B cells) or `CD3D`
(T cells) and re-draw. Then do the same in Section 3.5 for the Atera whole slide. The B-cell
map is the interesting one: on Visium it is almost blank, on Atera it is a small number of
very bright cells. Explain in one sentence why the same biology looks so different, and say
which of the two pictures you would put in a paper about tertiary lymphoid structures.

**2. Change the Leiden resolution.** Section 2.7 uses `resolution=0.6`. Try `0.3` and `1.2`,
and re-run the boundary plot in 2.8 each time. How many clusters do you get? Does the duct
wall stay one cluster, or split? There is no correct resolution &mdash; describe what you are
choosing between, and how the marker dotplot helps you decide.

**3. Choose a platform, and defend it.** You want to know whether T cells are **excluded**
from tumour nests or **infiltrating** them, in 40 patient samples, and you have budget for
one technology. Which do you pick, and why? Argue it in terms of the specific rows of the
Section 4 table &mdash; unit size, panel, counts per unit &mdash; not in general terms. Then
say what you would lose by that choice, and what a second, cheaper assay could recover.

**4. Move the crop.** Section 2.6 takes a 1 mm² window at (5500, 3000) µm. Pick a different
one &mdash; the whole slide runs to about 7,500 µm in x and 5,500 µm in y &mdash; and re-run
Sections 2.6 to 2.8. Find a window whose Leiden clusters look *different* from ours, and work
out from the dotplot what changed. (If a window comes back nearly empty, you have found a
hole in the section; that is informative too.)

**Going further.** Everything here used the vendor's segmentation. Both `cell_boundaries` and
`nucleus_boundaries` are in the Xenium object &mdash; compare the two areas per cell, and ask
how much of a "cell's" expression profile is really coming from its neighbours. That question,
**transcript misassignment**, is one of the live methodological problems in the field.